In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

bronze_stream_df = spark.readStream.table("retail_lakehouse.bronze.orders_bronze")

silver_staging_df = (
    bronze_stream_df
        .withColumn("quantity", col("quantity").cast("int"))
        .withColumn("unit_price", col("unit_price").cast("double"))
        .withColumn("order_date", to_date(col("order_date")))
        .withColumn(
            "order_status",
            when(
                upper(col("order_status")) == "DELIVERD",
                "DELIVERED"
            )
            .when(
                upper(col("order_status")) == "DELIVERED",
                "DELIVERED"
            )
            .when(
                upper(col("order_status")) == "PENDNG",
                "PENDING"
            )
            .when(
                upper(col("order_status")) == "PENDING",
                "PENDING"
            )
            .when(
                upper(col("order_status")) == "CNCLD",
                "CANCELLED"
            )
            .when(
                upper(col("order_status")) == "CANCELLED",
                "CANCELLED"
            )
            .otherwise(upper(col("order_status")))
        )
)

valid_condition = (
    col("order_id").isNotNull()
    & col("customer_id").isNotNull()
    & col("product_id").isNotNull()
    & col("city").isNotNull()
    & col("payment_method").isNotNull()
    & col("quantity").cast("int").isNotNull()
    & (col("quantity").cast("int") > 0)
    & col("unit_price").cast("double").isNotNull()
    & (col("unit_price").cast("double") > 0)
)

validated_df = silver_staging_df.filter(valid_condition)

quarantine_df = silver_staging_df.filter(~valid_condition)

silver_table = "retail_lakehouse.silver.orders_silver"

quarantine_table = "retail_lakehouse.silver.orders_quarantine"

silver_stream = (
    validated_df.writeStream
        .format("delta")
        .option(
            "checkpointLocation",
            "/Volumes/retail_lakehouse/bronze/raw_files/checkpoints/orders_silver/"
        )
        .trigger(availableNow=True)
        .toTable(silver_table)
)

quarantine_stream = (
    quarantine_df.writeStream
        .format("delta")
        .option(
            "checkpointLocation",
            "/Volumes/retail_lakehouse/bronze/raw_files/checkpoints/orders_quarantine/"
        )
        .trigger(availableNow=True)
        .toTable(quarantine_table)
)